# Step 2 — Feature Engineering

We turn the raw SteamSpy tag data into a numeric feature matrix that a cosine similarity model can use.

**Strategy — TF-IDF on tags:**
- **TF** (term frequency): how many votes a tag got for this game relative to the game's total votes
- **IDF** (inverse document frequency): tags that appear on every game (e.g. "Action") are less informative — we down-weight them
- Result: niche tags like "Metroidvania" or "Soulslike" carry more weight than "Indie"

**Output**: `../data/features.pkl` — a dict with the feature matrix, game DataFrame, and tag list.

In [26]:
import os
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import TfidfTransformer

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
os.chdir(root)

df = pd.read_csv("data/steamspy_games.csv")
print(f"Loaded {len(df)} games")
df.head(3)

Loaded 4995 games


,Unnamed: 0.1,Unnamed: 0,appid,name,developer,publisher,owners,average_playtime,median_playtime,ccu,tags
0,0,0,730,Counter-Strike: Global Offensive,Valve,Valve,150000000,0,0,1013936,"{""FPS"": 91172, ""Shooter"": 65634, ""Multiplayer""..."
1,1,1,1172470,Apex Legends,Respawn,Electronic Arts,150000000,0,0,124262,"{""Free to Play"": 2232, ""Battle Royale"": 1511, ..."
2,2,2,578080,PUBG: BATTLEGROUNDS,PUBG Corporation,"KRAFTON, Inc.",150000000,0,0,314682,"{""Survival"": 14893, ""Shooter"": 12788, ""Battle ..."


In [27]:
df

,Unnamed: 0.1,Unnamed: 0,appid,name,developer,publisher,owners,average_playtime,median_playtime,ccu,tags
0,0,0,730,Counter-Strike: Global Offensive,Valve,Valve,150000000,0,0,1013936,"{""FPS"": 91172, ""Shooter"": 65634, ""Multiplayer""..."
1,1,1,1172470,Apex Legends,Respawn,Electronic Arts,150000000,0,0,124262,"{""Free to Play"": 2232, ""Battle Royale"": 1511, ..."
2,2,2,578080,PUBG: BATTLEGROUNDS,PUBG Corporation,"KRAFTON, Inc.",150000000,0,0,314682,"{""Survival"": 14893, ""Shooter"": 12788, ""Battle ..."
3,3,3,1623730,Palworld,Pocketpair,Pocketpair,75000000,0,0,18028,"{""Open World"": 1508, ""Survival"": 1382, ""Creatu..."
4,4,4,440,Team Fortress 2,Valve,Valve,75000000,0,0,43819,"{""Free to Play"": 62968, ""Hero Shooter"": 61037,..."
...,...,...,...,...,...,...,...,...,...,...,...
4990,4990,4990,306660,Ultimate General: Gettysburg,Game-Labs,Game-Labs,350000,0,0,11,"{""Strategy"": 181, ""Historical"": 131, ""Simulati..."
4991,4991,4991,335430,Grimoire: Manastorm,Omniconnection,Omniconnection,350000,0,0,1,"{""Action"": 53, ""Shooter"": 48, ""Indie"": 46, ""Ma..."
4992,4992,4992,1462810,WRC 10 FIA World Rally Championship,KT Racing,Nacon,350000,0,0,44,"{""Racing"": 184, ""Automobile Sim"": 156, ""Sports..."
4993,4993,4993,743390,DISTRAINT 2,Jesse Makkonen,Jesse Makkonen,350000,0,0,1,"{""Psychological Horror"": 188, ""Horror"": 180, ""..."


## Build the tag vocabulary

Collect all unique tags and filter out very rare ones (appearing in fewer than 10 games) to keep the matrix manageable.

In [28]:
from collections import Counter

tag_game_count = Counter()
for tags_str in df["tags"]:
    tags = json.loads(tags_str)
    tag_game_count.update(tags.keys())

MIN_GAMES = 10
vocab = [tag for tag, count in tag_game_count.items() if count >= MIN_GAMES]
tag_to_idx = {tag: i for i, tag in enumerate(vocab)}

print(f"Total unique tags: {len(tag_game_count)}")
print(f"Tags appearing in >= {MIN_GAMES} games: {len(vocab)}")

Total unique tags: 443
Tags appearing in >= 10 games: 380


## Build raw tag count matrix

Shape: `(n_games, n_tags)`. Each cell is the vote count that tag received for that game.

In [29]:
rows, cols, data = [], [], []

for game_idx, tags_str in enumerate(df["tags"]):
    tags = json.loads(tags_str)
    for tag, votes in tags.items():
        if tag in tag_to_idx and votes > 0:
            rows.append(game_idx)
            cols.append(tag_to_idx[tag])
            data.append(float(votes))

count_matrix = csr_matrix(
    (data, (rows, cols)),
    shape=(len(df), len(vocab))
)

print(f"Matrix shape: {count_matrix.shape}")
print(f"Sparsity: {1 - count_matrix.nnz / (count_matrix.shape[0] * count_matrix.shape[1]):.1%}")

Matrix shape: (4995, 380)
Sparsity: 95.3%


## Apply TF-IDF and L2 normalize

After TF-IDF, we L2-normalize each game's vector so cosine similarity is just a dot product.

In [30]:
transformer = TfidfTransformer(smooth_idf=True, sublinear_tf=True)
tfidf_matrix = transformer.fit_transform(count_matrix)
feature_matrix = normalize(tfidf_matrix, norm="l2")

print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"Type: {type(feature_matrix)}")

Feature matrix shape: (4995, 380)
Type: <class 'scipy.sparse._csr.csr_matrix'>


## Sanity check — inspect one game's top features

In [31]:
def top_tags_for_game(name: str, n: int = 15):
    matches = df[df["name"].str.lower() == name.lower()]
    if matches.empty:
        matches = df[df["name"].str.lower().str.contains(name.lower())]
    if matches.empty:
        print(f"Game not found: {name}")
        return
    idx = matches.index[0]
    game_vec = feature_matrix[idx].toarray()[0]
    top_idx = np.argsort(game_vec)[::-1][:n]
    print(f"Top TF-IDF tags for '{df.loc[idx, 'name']}'")
    for i in top_idx:
        if game_vec[i] > 0:
            print(f"  {vocab[i]}: {game_vec[i]:.4f}")

top_tags_for_game("Counter-Strike 2")
print()
top_tags_for_game("The Witcher 3")

Game not found: Counter-Strike 2

Top TF-IDF tags for 'The Witcher 3: Wild Hunt'
  Mature: 0.3083
  Magic: 0.2935
  Medieval: 0.2877
  Multiple Endings: 0.2855
  Nudity: 0.2843
  Dark Fantasy: 0.2834
  Choices Matter: 0.2674
  Dark: 0.2405
  Action RPG: 0.2387
  Fantasy: 0.2052
  Third Person: 0.1948
  Open World: 0.1891
  Story Rich: 0.1834
  Sandbox: 0.1804
  Great Soundtrack: 0.1792


## Save feature matrix

In [32]:
# Store only the columns we need at query time
games_slim = df[["appid", "name", "developer", "publisher", "owners", "tags"]].copy()

# Add a human-readable top_tags column for display in API responses
def top_tags_str(tags_json: str, n: int = 5) -> str:
    tags = json.loads(tags_json)
    top = sorted(tags.items(), key=lambda x: x[1], reverse=True)[:n]
    return ",".join(t for t, _ in top)

games_slim["top_tags"] = games_slim["tags"].apply(top_tags_str)

payload = {
    "games": games_slim.reset_index(drop=True),
    "matrix": feature_matrix,
    "vocab": vocab,
    "tfidf_transformer": transformer,
}

with open("data/features.pkl", "wb") as f:
    pickle.dump(payload, f)

print("Saved data/features.pkl")
print(f"  Games: {len(games_slim)}")
print(f"  Features (tags): {len(vocab)}")

Saved data/features.pkl
  Games: 4995
  Features (tags): 380


---
## Summary

- `data/features.pkl` — TF-IDF tag feature matrix ready for cosine similarity

**Next**: `03_model_exploration.ipynb` — test the recommender interactively before wiring up the API.